# 📊 Análise de Dados — Tótem Inteligente Inclusivo (Sprint 4)

**Projeto:** Challenge Flexmedia | FIAP  
**Sprint:** 4 — Entrega Final  

Este notebook realiza a análise exploratória completa dos dados coletados pelo Tótem, incluindo:
- Interações de sensor
- Previsões do modelo ML
- Sessões e mensagens do chatbot
- Eventos de visão computacional
- Eventos de reconhecimento de voz


## 1. Imports e Conexão com o Banco

In [ ]:
import sys
import json
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Ajusta path para importar módulos do sprint4
NOTEBOOK_DIR = Path().resolve()
SPRINT4_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else NOTEBOOK_DIR
sys.path.insert(0, str(SPRINT4_DIR))

from database.db import get_conn, init_db

DB_PATH = SPRINT4_DIR / "database" / "totem.db"
ARTIFACTS_DIR = SPRINT4_DIR / "ml" / "artifacts"

print(f"DB: {DB_PATH}")
print(f"DB existe: {DB_PATH.exists()}")


## 2. Carregamento dos Dados

In [ ]:
conn = get_conn(DB_PATH)
init_db(conn)

interactions = pd.read_sql_query("SELECT * FROM interactions", conn)
predictions = pd.read_sql_query("SELECT * FROM predictions", conn)
chat_sessions = pd.read_sql_query("SELECT * FROM chat_sessions", conn)
chat_messages = pd.read_sql_query("SELECT * FROM chat_messages", conn)
vision_events = pd.read_sql_query("SELECT * FROM vision_events", conn)
voice_events = pd.read_sql_query("SELECT * FROM voice_events", conn)

conn.close()

print(f"Interactions: {len(interactions):,} registros")
print(f"Predictions: {len(predictions):,} registros")
print(f"Chat sessions: {len(chat_sessions):,} registros")
print(f"Chat messages: {len(chat_messages):,} registros")
print(f"Vision events: {len(vision_events):,} registros")
print(f"Voice events: {len(voice_events):,} registros")


## 3. Pré-processamento

In [ ]:
# Conversão de timestamps
interactions["event_timestamp"] = pd.to_datetime(interactions["event_timestamp"], errors="coerce", utc=True)
interactions["hour"] = interactions["event_timestamp"].dt.hour
interactions["weekday"] = interactions["event_timestamp"].dt.day_name()
interactions["date"] = interactions["event_timestamp"].dt.date

# Classe de engajamento
def dur_class(d):
    d = int(d) if pd.notnull(d) else 0
    if d <= 5: return "quick"
    if d <= 20: return "normal"
    return "engaged"

interactions["engagement"] = interactions["duration_s"].apply(dur_class)

# Vision
vision_events["detected_at"] = pd.to_datetime(vision_events["detected_at"], errors="coerce", utc=True)
vision_events["hour"] = vision_events["detected_at"].dt.hour

# Chat
chat_messages["created_at"] = pd.to_datetime(chat_messages["created_at"], errors="coerce", utc=True)

print("Pré-processamento concluído ✅")
interactions[["hour", "duration_s", "presence", "touch", "voice_detected", "engagement"]].describe()


## 4. Análise de Interações de Sensor

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Análise de Interações — Sensor (Sprint 4)", fontsize=16, fontweight="bold")

# 4.1 Eventos por hora
ax = axes[0, 0]
by_hour = interactions.groupby("hour")["id"].count()
ax.bar(by_hour.index, by_hour.values, color="#4F8EF7", edgecolor="white")
ax.set_title("Eventos por Hora do Dia")
ax.set_xlabel("Hora")
ax.set_ylabel("Nº de eventos")
ax.set_xticks(range(0, 24))

# 4.2 Distribuição de engajamento
ax = axes[0, 1]
eng_colors = {"quick": "#F7844F", "normal": "#4FF7A0", "engaged": "#C84FF7"}
eng_dist = interactions["engagement"].value_counts()
ax.bar(eng_dist.index, eng_dist.values,
       color=[eng_colors.get(k, "#4F8EF7") for k in eng_dist.index], edgecolor="white")
ax.set_title("Distribuição de Engajamento")
ax.set_ylabel("Nº de interações")

# 4.3 Presença por hora
ax = axes[1, 0]
pres_hour = interactions.groupby("hour")["presence"].sum()
ax.plot(pres_hour.index, pres_hour.values, color="#F7C84F", marker="o", linewidth=2)
ax.fill_between(pres_hour.index, pres_hour.values, alpha=0.3, color="#F7C84F")
ax.set_title("Ativações de Presença por Hora")
ax.set_xlabel("Hora")
ax.set_xticks(range(0, 24))

# 4.4 Categorias de conteúdo
ax = axes[1, 1]
if "content_category" in interactions.columns:
    cat_dist = interactions["content_category"].value_counts()
    ax.barh(cat_dist.index, cat_dist.values, color="#4FF7A0", edgecolor="white")
    ax.set_title("Categorias de Conteúdo Acessadas")
    ax.set_xlabel("Nº de acessos")

plt.tight_layout()
plt.savefig("sensor_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: sensor_analysis.png")


## 5. Análise do Modelo ML

In [ ]:
# Carrega métricas
metrics_path = ARTIFACTS_DIR / "metrics.json"
metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

print("=== Métricas do Modelo ===")
print(f"Modelo: {metrics.get('model_name', '-')}")
print(f"Versão: {metrics.get('model_version', '-')}")
print(f"Linhas no treino: {metrics.get('rows_used', '-')}")
print()
print(f"Baseline Accuracy:     {metrics.get('baseline_accuracy', 0):.4f}")
print(f"Baseline F1-macro:     {metrics.get('baseline_f1_macro', 0):.4f}")
print(f"RF Accuracy (FULL):    {metrics.get('rf_full_accuracy', 0):.4f}")
print(f"RF F1-macro (FULL):    {metrics.get('rf_full_f1_macro', 0):.4f}")
print(f"RF Accuracy (NO dur):  {metrics.get('rf_no_duration_accuracy', 0):.4f}")
print(f"RF F1-macro (NO dur):  {metrics.get('rf_no_duration_f1_macro', 0):.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Machine Learning — Classificação de Engajamento", fontsize=14, fontweight="bold")

# 5.1 Comparação de modelos
ax = axes[0]
model_names = ["Baseline\nAccuracy", "Baseline\nF1-macro", "RF FULL\nAccuracy", "RF FULL\nF1-macro"]
model_values = [
    metrics.get("baseline_accuracy", 0),
    metrics.get("baseline_f1_macro", 0),
    metrics.get("rf_full_accuracy", 0),
    metrics.get("rf_full_f1_macro", 0),
]
colors = ["#aaaaaa", "#aaaaaa", "#4F8EF7", "#4F8EF7"]
bars = ax.bar(model_names, model_values, color=colors, edgecolor="white")
ax.set_ylim(0, 1.05)
ax.set_title("Comparação: Baseline vs RandomForest")
ax.set_ylabel("Score")
for bar, val in zip(bars, model_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")

# 5.2 Distribuição das previsões
ax = axes[1]
if not predictions.empty and "pred_label" in predictions.columns:
    pred_colors = {"quick": "#F7844F", "normal": "#4FF7A0", "engaged": "#C84FF7"}
    pred_dist = predictions["pred_label"].value_counts()
    ax.pie(pred_dist.values, labels=pred_dist.index, autopct="%1.1f%%",
           colors=[pred_colors.get(k, "#4F8EF7") for k in pred_dist.index],
           startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
    ax.set_title("Distribuição das Classes Previstas")

plt.tight_layout()
plt.savefig("ml_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Análise do Chatbot

In [ ]:
user_msgs = chat_messages[chat_messages["role"] == "user"].copy()

print(f"Sessões totais: {len(chat_sessions)}")
print(f"Mensagens totais: {len(chat_messages)}")
print(f"Mensagens de usuário: {len(user_msgs)}")
print(f"Média mensagens/sessão: {chat_sessions['total_messages'].mean():.1f}")

print("\nTop 5 intenções detectadas:")
if "intent" in user_msgs.columns:
    print(user_msgs["intent"].value_counts().head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Analytics do Chatbot — NLP Local (Sprint 4)", fontsize=14, fontweight="bold")

# 6.1 Intenções
ax = axes[0]
if "intent" in user_msgs.columns:
    intent_dist = user_msgs["intent"].value_counts()
    ax.barh(intent_dist.index, intent_dist.values, color="#4F8EF7", edgecolor="white")
    ax.set_title("Intenções Detectadas")
    ax.set_xlabel("Nº de ocorrências")

# 6.2 Modos de entrada
ax = axes[1]
if "input_mode" in user_msgs.columns:
    mode_dist = user_msgs["input_mode"].value_counts()
    ax.bar(mode_dist.index, mode_dist.values, color=["#4FF7A0", "#F7844F", "#C84FF7"][:len(mode_dist)], edgecolor="white")
    ax.set_title("Modos de Entrada")
    ax.set_ylabel("Nº de mensagens")

# 6.3 Distribuição de confiança
ax = axes[2]
if "confidence" in user_msgs.columns:
    conf_data = user_msgs["confidence"].dropna()
    ax.hist(conf_data, bins=15, color="#F7C84F", edgecolor="white")
    ax.axvline(conf_data.mean(), color="#F7844F", linestyle="--", linewidth=2,
               label=f"Média: {conf_data.mean():.2f}")
    ax.set_title("Distribuição de Confiança do NLP")
    ax.set_xlabel("Confiança")
    ax.legend()

plt.tight_layout()
plt.savefig("chatbot_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Análise de Visão Computacional

In [ ]:
with_person = vision_events[vision_events["person_detected"] == 1]
print(f"Total frames: {len(vision_events)}")
print(f"Com pessoa: {len(with_person)} ({len(with_person)/len(vision_events)*100:.1f}%)")
print(f"Score médio de atenção: {with_person['attention_score'].mean():.3f}")
print()
print("Distribuição por faixa etária:")
print(with_person["age_group"].value_counts())
print()
print("Distribuição por emoção:")
print(with_person["emotion"].value_counts())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Visão Computacional — Detecção e Análise (Sprint 4)", fontsize=14, fontweight="bold")

# 7.1 Faixa etária
ax = axes[0, 0]
age_dist = with_person["age_group"].value_counts()
age_colors = {"crianca": "#4FF7A0", "jovem": "#4F8EF7", "adulto": "#C84FF7", "idoso": "#F7C84F"}
ax.bar(age_dist.index, age_dist.values,
       color=[age_colors.get(k, "#4F8EF7") for k in age_dist.index], edgecolor="white")
ax.set_title("Distribuição por Faixa Etária")
ax.set_ylabel("Nº de detecções")

# 7.2 Emoção
ax = axes[0, 1]
emo_dist = with_person["emotion"].value_counts()
emo_colors = {"neutro": "#aaaaaa", "feliz": "#4FF7A0", "curioso": "#4F8EF7", "confuso": "#F7844F"}
ax.bar(emo_dist.index, emo_dist.values,
       color=[emo_colors.get(k, "#4F8EF7") for k in emo_dist.index], edgecolor="white")
ax.set_title("Distribuição por Emoção")
ax.set_ylabel("Nº de detecções")

# 7.3 Detecção por hora
ax = axes[1, 0]
hour_pres = vision_events[vision_events["person_detected"] == 1].groupby("hour")["id"].count()
ax.plot(hour_pres.index, hour_pres.values, color="#F7C84F", marker="o", linewidth=2)
ax.fill_between(hour_pres.index, hour_pres.values, alpha=0.3, color="#F7C84F")
ax.set_title("Detecção de Pessoa por Hora")
ax.set_xlabel("Hora")

# 7.4 Zonas de interação
ax = axes[1, 1]
zone_dist = with_person["zone"].value_counts()
ax.barh(zone_dist.index, zone_dist.values, color="#4F8EF7", edgecolor="white")
ax.set_title("Zonas de Interação Mais Visitadas")
ax.set_xlabel("Nº de detecções")

plt.tight_layout()
plt.savefig("vision_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Análise de Reconhecimento de Voz

In [ ]:
processed_voice = voice_events[voice_events["processed"] == 1]
print(f"Total eventos de voz: {len(voice_events)}")
print(f"Transcrições bem-sucedidas: {len(processed_voice)} ({len(processed_voice)/len(voice_events)*100:.1f}%)")
print(f"Confiança média: {processed_voice['confidence'].mean():.3f}")
print(f"Duração média: {processed_voice['duration_ms'].mean():.0f}ms")
print()
print("Amostra de transcrições:")
for t in processed_voice["transcript"].dropna().head(8).values:
    print(f"  → "{t}"")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Reconhecimento de Voz (Sprint 4)", fontsize=14, fontweight="bold")

# 8.1 Distribuição de confiança
ax = axes[0]
conf_v = processed_voice["confidence"].dropna()
ax.hist(conf_v, bins=15, color="#4F8EF7", edgecolor="white")
ax.axvline(conf_v.mean(), color="#F7844F", linestyle="--", linewidth=2,
           label=f"Média: {conf_v.mean():.2f}")
ax.set_title("Distribuição de Confiança da Transcrição")
ax.set_xlabel("Confiança")
ax.legend()

# 8.2 Duração dos áudios
ax = axes[1]
dur_v = processed_voice["duration_ms"].dropna()
ax.hist(dur_v, bins=15, color="#4FF7A0", edgecolor="white")
ax.axvline(dur_v.mean(), color="#F7844F", linestyle="--", linewidth=2,
           label=f"Média: {dur_v.mean():.0f}ms")
ax.set_title("Distribuição de Duração dos Áudios")
ax.set_xlabel("Duração (ms)")
ax.legend()

plt.tight_layout()
plt.savefig("voice_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Resumo Final — Insights Integrados

In [ ]:
print("=" * 60)
print("  RESUMO EXECUTIVO — SPRINT 4")
print("=" * 60)

print(f"\n[SENSOR]")
print(f"  Eventos coletados:       {len(interactions):,}")
print(f"  Taxa de validade:        {interactions['is_valid'].mean()*100:.1f}%")
print(f"  Taxa de presença:        {interactions['presence'].mean()*100:.1f}%")
print(f"  Duração média:           {interactions['duration_s'].mean():.1f}s")

print(f"\n[MACHINE LEARNING]")
print(f"  Modelo:                  {metrics.get('model_name', '-')}")
print(f"  Acurácia (RF FULL):      {metrics.get('rf_full_accuracy', 0):.4f}")
print(f"  F1-macro (RF FULL):      {metrics.get('rf_full_f1_macro', 0):.4f}")
print(f"  Previsões gravadas:      {len(predictions):,}")

print(f"\n[CHATBOT]")
print(f"  Sessões iniciadas:       {len(chat_sessions)}")
print(f"  Mensagens totais:        {len(chat_messages)}")
print(f"  Média msgs/sessão:       {chat_sessions['total_messages'].mean():.1f}")

print(f"\n[VISÃO COMPUTACIONAL]")
print(f"  Frames analisados:       {len(vision_events):,}")
print(f"  Taxa de detecção:        {vision_events['person_detected'].mean()*100:.1f}%")
print(f"  Atenção média:           {with_person['attention_score'].mean():.3f}")

print(f"\n[VOZ]")
print(f"  Eventos de voz:          {len(voice_events)}")
print(f"  Taxa de sucesso:         {voice_events['processed'].mean()*100:.1f}%")
print(f"  Confiança média:         {processed_voice['confidence'].mean():.3f}")
print()
print("Pipeline 100% local — Sprint 4 concluída ✅")
